# T1.L3. Організація математичного моделювання
## Demo notebook

Мета: пройти повний відтворюваний workflow від config і data до results, summary та metadata.

## 1. Research question

**Як змінюється результативність умовної системи при зміні ресурсу або навантаження, і як організувати експеримент так, щоб його можна було повторити?**

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "lessons" / "t1_l3").exists():
    for candidate in [ROOT, *ROOT.parents]:
        if (candidate / "lessons" / "t1_l3").exists():
            ROOT = candidate
            break

sys.path.insert(0, str(ROOT))
LESSON = ROOT / "lessons" / "t1_l3"

from lessons.t1_l3.src.model import deterministic_response
from lessons.t1_l3.src.experiment import (
    load_config,
    run_experiment,
    summarize_results,
    build_metadata,
    save_outputs,
)

## 2. Input data and configuration

In [ ]:
config = load_config(LESSON / "experiment_config.json")
scenarios = pd.read_csv(LESSON / "data" / "scenarios.csv")
config, scenarios

## 3. Verification

Для baseline:

\[
Y=20+1.8\cdot40-1.2\cdot50=32.
\]

In [ ]:
manual = 20 + 1.8 * 40 - 1.2 * 50
python_value = float(deterministic_response(40, 50))
manual, python_value

In [ ]:
assert manual == 32
assert python_value == 32

## 4. Run the experiment

In [ ]:
results = run_experiment(config, scenarios)
summary = summarize_results(results)
metadata = build_metadata(config, scenarios)

summary

## 5. Experiment identity

In [ ]:
metadata

## 6. Save reproducible outputs

In [ ]:
save_outputs(results, summary, metadata, LESSON / "outputs")
sorted(p.name for p in (LESSON / "outputs").iterdir())

## 7. Visualization connected to the research question

In [ ]:
plot_df = summary.sort_values("mean_observed")
yerr = [
    plot_df["mean_observed"] - plot_df["p10"],
    plot_df["p90"] - plot_df["mean_observed"],
]

ax = plot_df.plot(
    x="scenario_id",
    y="mean_observed",
    kind="bar",
    yerr=yerr,
    capsize=4,
    legend=False,
    figsize=(9, 5),
)
ax.set_title("Scenario comparison: mean response and P10–P90 interval")
ax.set_xlabel("Scenario")
ax.set_ylabel("Modeled response")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 8. Interpretation

Зверніть увагу:

- `resource_high` і `load_low` можуть дати однаковий детермінований результат;
- це **не означає**, що механізм зміни однаковий;
- raw result, summary і metadata — різні рівні представлення експерименту;
- experiment ID і seed допомагають traceability, але для повної відтворюваності також потрібна версія коду та software environment.

## 9. Research transfer

Сформулюйте для власного дисертаційного дослідження:

1. research question;
2. input data;
3. model;
4. config;
5. scenarios;
6. verification;
7. outputs;
8. metadata;
9. Git state/commit.